# Tensor Decomposition
*Thomas Dooms*

In tutorial 1, we decomposed the bilinear interaction tensor using eigendecomposition. This gives us interpretable features *per output class*, but there are some issues with this. The most important one is orthogonality; you might have overlapping structures in input space which behave differently. The second is somewhat of a consequence where eigendecompositions often yield "superposed" features; we might hope that a 6 decomposes into a few edge-detectors but that generally doesn't happen. If the model shares structure across classes, eigendecomposition can't see it.

Here we take a different approach. Instead of decomposing per class, we decompose the *full* third-order tensor into a set of shared **neurons**, each with explicit input patterns and output weights. The result is a set of interpretable building blocks that the model combines to classify digits.

### Setup

In [1]:
import ssl, certifi
ssl._create_default_https_context = lambda: ssl.create_default_context(cafile=certifi.where())

In [3]:
%load_ext autoreload
%autoreload 2

import torch
import plotly.express as px
from plotly.subplots import make_subplots

from image import Model, MNIST
from image.sparse import Model as Sparse
from kornia.augmentation import RandomGaussianNoise

device = "cpu"
train, test = MNIST(train=True, device=device), MNIST(train=False, device=device)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Training the original model

As in tutorial 1, we train a bilinear model with Gaussian noise augmentation to prevent overfitting on individual pixels.

In [4]:
model = Model.from_config(epochs=20).to(device)
model.fit(train, test, RandomGaussianNoise(std=0.4))

train/loss: 0.149, train/acc: 0.956, val/loss: 0.113, val/acc: 0.967: 100%|██████████| 20/20 [00:07<00:00,  2.83it/s]


,train/loss,train/acc,val/loss,val/acc
0,1.131199,0.686456,0.451672,0.8831
1,0.450729,0.873990,0.324314,0.9154
2,0.389520,0.889261,0.287193,0.9226
3,0.347364,0.900340,0.250061,0.9311
4,0.298244,0.914113,0.208244,0.9401
5,0.260067,0.925276,0.175644,0.9497
6,0.234229,0.934065,0.160026,0.9531
7,0.211315,0.939638,0.146847,0.9579
8,0.196227,0.943326,0.138285,0.9600
9,0.186106,0.946575,0.134037,0.9630


### From eigendecomposition to tensor decomposition

Recall that the bilinear model computes $\text{output}_c = \sum_{i,j} B_{c,i,j}\, x_i\, x_j$, where $B$ is the third-order interaction tensor. Eigendecomposition slices $B$ along the class axis and decomposes each slice independently.

We can instead factorize the *entire* tensor at once into a new bilinear layer:
$$B_{c,i,j} \approx \sum_{r=1}^{R} L_{i,r}\, R_{j,r}\, D_{c,r}$$

Each component $r$ is a **neuron** with three parts:
- $L_r \in \mathbb{R}^{784}$, the left input pattern
- $R_r \in \mathbb{R}^{784}$, the right input pattern  
- $D_r \in \mathbb{R}^{10}$, the output weights over classes

The neuron's activation on input $x$ is $(L_r^\top x)(R_r^\top x)$. Using the identity $ab = \tfrac{1}{4}(a+b)^2 - \tfrac{1}{4}(a-b)^2$, this becomes:
$$\tfrac{1}{4}\big((L_r + R_r)^\top x\big)^2 - \tfrac{1}{4}\big((L_r - R_r)^\top x\big)^2$$

So each neuron naturally decomposes into a **positive pattern** $L_r + R_r$ (matching it increases activation) and a **negative pattern** $L_r - R_r$ (matching it decreases activation). These are directly analogous to eigenvectors with positive and negative eigenvalues.

### Fitting the decomposition

We fit a new bilinear layer by maximizing the cosine similarity between the original interaction tensor and our approximation. The `Sparse` model stores $L$, $R$, and $D$ as learnable parameters and is optimized with Muon.

In [5]:
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm

sparse = Sparse.from_config(rank=64).to(device)

optimizer = torch.optim.Muon(sparse.parameters(), lr=0.02, momentum=0.95)
scheduler = CosineAnnealingLR(optimizer, T_max=200)

torch.set_grad_enabled(True)
for _ in (pbar := tqdm(range(200))):
    loss = 1 - sparse.similarity(model)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    scheduler.step()
    pbar.set_description(f"loss: {loss.item():.4f}")
torch.set_grad_enabled(False)

with torch.no_grad():
    orig_acc = (model(test.x).argmax(-1) == test.y).float().mean()
    sparse_acc = (sparse(test.x).argmax(-1) == test.y).float().mean()
    print(f"Original: {orig_acc:.3f}, Sparse: {sparse_acc:.3f}")

loss: -0.0006: 100%|██████████| 200/200 [00:04<00:00, 43.73it/s]

Original: 0.967, Sparse: 0.967


### Visualizing neurons

Each neuron has a positive pattern ($L_r + R_r$), a negative pattern ($L_r - R_r$), and output weights ($D_r$). The `decompose` method returns these normalized and sorted by importance (the product of the factor norms).

In [6]:
plus, minus, down, sigma = sparse.decompose()
plus, minus, down, sigma = plus.cpu(), minus.cpu(), down.cpu(), sigma.cpu()

k = 8
fig = make_subplots(rows=3, cols=k, row_titles=["l+r", "l-r", "logits"], vertical_spacing=0.08)

for i in range(k):
    params = dict(showscale=False, colorscale="RdBu", zmid=0)
    fig.add_heatmap(z=plus[:, i].view(28, 28).flip(0), **params, row=1, col=i+1)
    fig.add_heatmap(z=minus[:, i].view(28, 28).flip(0), **params, row=2, col=i+1)
    fig.add_bar(y=down[:, i], marker_color=["gray"]*10, showlegend=False, row=3, col=i+1)

fig.update_xaxes(visible=False).update_yaxes(visible=False)
fig.update_layout(width=k*110, height=360, margin=dict(l=40, r=0, b=0, t=0), template="plotly_white")

In [7]:
px.bar(sigma, template="plotly_white", labels=dict(index="component", value="sigma"))

### Exercises

The goal of this part is to implement a decomposition that is more in line with our requirements. This is where you try to find what kinds of priors and structure work well. The code above provides a skeleton with the essentials: the `Sparse` model, the reconstruction loop, and the visualization. There's no right answer and this is a hard task to get right.

## Easy Wins: Estimating similarity and completeness of digit-specifc eigenvectors.

### Compare eigenvectors across digits:
    - for each digit, take top few eigenvectors, sorted by eigenvalue.
    - flatten them.
    - compute cosine similarity between the digits.

### Measure the completeness --- number of eigenvectors required for >90\%.

In [8]:
from einops import einsum

l, r = model.w_l[0], model.w_r[0]

b = einsum(
    model.w_u, l, r, model.w_e, model.w_e,
    "cls out, out hid1, out hid2, hid1 in1, hid2 in2 -> cls in1 in2"
)

b = 0.5 * (b + b.mT)

In [9]:
vals, vecs = torch.linalg.eigh(b)

# sort by absolute eigenvalue magnitude
idx = torch.argsort(vals.abs(), dim=-1, descending=True)

vals_sorted = torch.gather(vals, 1, idx)

vecs_sorted = torch.gather(
    vecs,
    2,
    idx[:, None, :].expand(-1, vecs.shape[1], -1)
)

In [10]:
top_k = 5

top_vecs = []
names = []

for c in range(10):
    idx = torch.argsort(vals[c].abs(), descending=True)[:top_k]
    for j in idx:
        v = vecs[c, :, j]
        v = v / (v.norm() + 1e-8)
        top_vecs.append(v)
        names.append(f"{c}:{j.item()}")

top_vecs = torch.stack(top_vecs)  # [10*top_k, 784]

sim = top_vecs @ top_vecs.T

px.imshow(
    sim.cpu(),
    color_continuous_midpoint=0,
    color_continuous_scale="RdBu",
    x=names,
    y=names,
    title="Cosine similarity between top eigenvectors across digits"
)

In [11]:
abs_vals = vals.abs()

# sort descending by magnitude
sorted_vals, _ = torch.sort(abs_vals, dim=1, descending=True)

cum = sorted_vals.cumsum(dim=1)
frac = cum / (sorted_vals.sum(dim=1, keepdim=True) + 1e-8)

px.line(
    frac.cpu().T,
    template="plotly_white",
    labels=dict(x="number of eigenvectors", y="fraction of absolute eigenvalue mass"),
    title="How many eigenvectors explain each digit?"
)

## Cross - Completeness:
- Predict how much of a digit_i can be explained by another digit_j's eigenvectors.
** helps us predict how much correlation exists between "features" of each digit.

In [12]:
import torch
import plotly.express as px

# Sort eigenvectors by descending absolute eigenvalue
idx = torch.argsort(vals.abs(), dim=1, descending=True)

vals_sorted = torch.gather(vals, 1, idx)

vecs_sorted = torch.gather(
    vecs,
    dim=2,
    index=idx[:, None, :].expand(-1, vecs.shape[1], -1)
)

In [13]:
def cross_class_eigen_completeness(b, vecs_sorted, k=20, eps=1e-8):
    """
    b: [C, P, P]
        class interaction matrices
    vecs_sorted: [C, P, P]
        eigenvectors sorted by abs eigenvalue descending
    k:
        number of top eigenvectors from source class to use

    returns:
        completeness: [C, C]
        completeness[target, source]
    """
    C, P, _ = b.shape
    completeness = torch.zeros(C, C, device=b.device)

    for target in range(C):
        B_t = b[target]
        denom = (B_t ** 2).sum() + eps

        for source in range(C):
            U = vecs_sorted[source, :, :k]  # [P, k]

            # Project B_t into source eigenvector subspace:
            # B_hat = U U^T B_t U U^T
            B_proj_small = U.T @ B_t @ U       # [k, k]
            B_hat = U @ B_proj_small @ U.T     # [P, P]

            completeness[target, source] = (B_hat ** 2).sum() / denom

    return completeness

In [14]:
k = 20
comp = cross_class_eigen_completeness(b, vecs_sorted, k=k)

px.imshow(
    comp.detach().cpu(),
    color_continuous_scale="Viridis",
    labels=dict(x="source eigenvectors", y="target class", color="completeness"),
    title=f"Cross-class eigen-completeness using top-{k} source eigenvectors",
)

In [18]:
for k in [1, 2, 5, 10, 20, 50]:
    comp = cross_class_eigen_completeness(b, vecs_sorted, k=k)
    fig = px.imshow(
        comp.detach().cpu(),
        color_continuous_scale="Viridis",
        labels=dict(x="source eigenvectors", y="target class", color="completeness"),
        title=f"Cross-class eigen-completeness using top-{k} source eigenvectors",
    )
    fig.show()

In [19]:
ks = [1, 2, 3, 5, 10, 15, 20, 30, 40, 50]

diag_means = []
offdiag_means = []

for k in ks:
    comp = cross_class_eigen_completeness(b, vecs_sorted, k=k)

    diag = torch.diag(comp)
    offdiag = comp[~torch.eye(comp.shape[0], dtype=torch.bool, device=comp.device)]

    diag_means.append(diag.mean().item())
    offdiag_means.append(offdiag.mean().item())

import pandas as pd

df = pd.DataFrame({
    "k": ks,
    "within-class": diag_means,
    "cross-class": offdiag_means,
})

px.line(
    df,
    x="k",
    y=["within-class", "cross-class"],
    markers=True,
    template="plotly_white",
    labels={"value": "eigen-completeness", "variable": "basis type"},
    title="Within-class vs cross-class eigen-completeness as k increases"
)

We compare within-class and cross-class eigen-completeness as a function of the number of source eigenvectors k. Within-class completeness rises rapidly, reaching nearly complete reconstruction by k≈10, indicating that each digit’s quadratic interaction matrix is effectively low-rank in its own eigenbasis. In contrast, cross-class completeness grows more gradually: the top few eigenvectors from another digit explain little of the target class, but completeness increases sharply between k=10 and k=20, and approaches within-class completeness by k=30. This suggests that the strongest eigenmodes are more class-specific, while a broader set of moderately important modes forms a shared visual subspace across digit classes.

## Using a global shared basis and testing reconstruction

 The previous results suggest that it should be possible to not just predict components with class scores but a more elegant decomposition is possible if all the digit classifiers use the same squared visual features --- just with different class weights

In [20]:
# b: [10, 784, 784], symmetrized B tensor

C, P, _ = b.shape

# Build a global interaction-energy matrix
G = torch.zeros(P, P, device=b.device)

for c in range(C):
    G += b[c] @ b[c].T

G = 0.5 * (G + G.mT)

# Eigenvectors of G define a shared visual basis
global_vals, global_vecs = torch.linalg.eigh(G)

idx = torch.argsort(global_vals.abs(), descending=True)
global_vals = global_vals[idx]
global_vecs = global_vecs[:, idx]

In [21]:
def shared_diagonal_reconstruction(b, global_vecs, R):
    V = global_vecs[:, :R]  # [P, R]

    # class-specific loading on each shared squared feature
    A = torch.einsum("pr,cij,jr->cr", V, b, V)

    # reconstruct B[c] ≈ sum_r A[c,r] v_r v_r^T
    b_hat = torch.einsum("cr,ir,jr->cij", A, V, V)

    completeness = b_hat.pow(2).sum(dim=(1, 2)) / (b.pow(2).sum(dim=(1, 2)) + 1e-8)

    return b_hat, A, completeness

In [22]:
Rs = [1, 2, 5, 10, 15, 20, 30, 40, 50]
means = []

for R in Rs:
    _, A, comp = shared_diagonal_reconstruction(b, global_vecs, R)
    means.append(comp.mean().item())

import pandas as pd
import plotly.express as px

df = pd.DataFrame({"R": Rs, "mean completeness": means})

px.line(
    df,
    x="R",
    y="mean completeness",
    markers=True,
    template="plotly_white",
    title="Shared diagonal eigenspace reconstruction of B"
)

In [23]:
R = 30
_, A, comp = shared_diagonal_reconstruction(b, global_vecs, R)

px.imshow(
    A.detach().cpu(),
    color_continuous_midpoint=0,
    color_continuous_scale="RdBu",
    labels=dict(x="shared eigenfeature", y="class", color="loading"),
    title="Class loadings on shared visual eigenfeatures"
)

In [31]:
import torch
import pandas as pd
import plotly.express as px

# b: [10, 784, 784]
b = 0.5 * (b + b.mT)
b = b.detach()

C, P, _ = b.shape

def frob_sq(x):
    return x.pow(2).sum(dim=(-2, -1))


# ----------------------------
# Per-class oracle eigenbasis
# ----------------------------
vals, vecs = torch.linalg.eigh(b)

idx = torch.argsort(vals.abs(), dim=1, descending=True)
vals_sorted = torch.gather(vals, 1, idx)

def per_class_oracle_completeness(vals_sorted, R):
    return vals_sorted[:, :R].pow(2).sum(dim=1) / (
        vals_sorted.pow(2).sum(dim=1) + 1e-8
    )


# ----------------------------
# Global shared basis
# ----------------------------
G = torch.zeros(P, P, device=b.device)

for c in range(C):
    G += b[c] @ b[c].T

G = 0.5 * (G + G.mT)

global_vals, global_vecs = torch.linalg.eigh(G)
gidx = torch.argsort(global_vals, descending=True)
global_vecs = global_vecs[:, gidx]

print(
    "orthonormal check:",
    ((global_vecs.T @ global_vecs - torch.eye(P, device=b.device)).abs().max()).item()
)


# ----------------------------
# Shared diagonal completeness
# B_c ≈ sum_r A[c,r] v_r v_r^T
# where A[c,r] = v_r^T B_c v_r
# ----------------------------
def shared_diag_completeness_from_V(b, V):
    # V: [P, R]

    # CORRECT: first V uses "ir", not "pr"
    A = torch.einsum("ir,cij,jr->cr", V, b, V)

    # Since {v_r v_r^T} are orthonormal under Frobenius,
    # explained energy is just sum_r A[c,r]^2.
    explained = A.pow(2).sum(dim=1)
    total = frob_sq(b) + 1e-8

    return explained / total


def random_shared_diag_completeness(b, R, n_trials=20):
    comps = []

    for _ in range(n_trials):
        Q, _ = torch.linalg.qr(torch.randn(P, R, device=b.device))
        comps.append(shared_diag_completeness_from_V(b, Q))

    return torch.stack(comps).mean(dim=0)


# ----------------------------
# Sweep
# ----------------------------
Rs = [1, 2, 3, 5, 10, 15, 20, 30, 40, 50]
rows = []

for R in Rs:
    oracle = per_class_oracle_completeness(vals_sorted, R).mean().item()

    V_shared = global_vecs[:, :R]
    shared = shared_diag_completeness_from_V(b, V_shared).mean().item()

    random = random_shared_diag_completeness(b, R, n_trials=20).mean().item()

    rows.append({"R": R, "completeness": oracle, "basis": "per-class eigenbasis"})
    rows.append({"R": R, "completeness": shared, "basis": "shared global basis"})
    rows.append({"R": R, "completeness": random, "basis": "random shared basis"})

df = pd.DataFrame(rows)

px.line(
    df,
    x="R",
    y="completeness",
    color="basis",
    markers=True,
    template="plotly_white",
    title="Shared diagonal basis vs baselines",
)

orthonormal check: 2.0265579223632812e-06


#### Implications:
- There is not one universal list of eigenvectors where each class only changes the scalar weight on each eigenvector.
- Classes may share a visual subspace, but not a shared diagonal eigenbasis.

In [32]:
def shared_subspace_completeness_from_V(b, V):
    """
    V: [P, R] orthonormal shared basis.
    Keeps the full class-specific core M_c = V^T B_c V.
    """
    M = torch.einsum("ir,cij,js->crs", V, b, V)          # [C, R, R]
    b_hat = torch.einsum("ir,crs,js->cij", V, M, V)      # [C, P, P]

    num = b_hat.pow(2).sum(dim=(1, 2))
    den = b.pow(2).sum(dim=(1, 2)) + 1e-8
    return num / den

In [33]:
rows = []
Rs = [1, 2, 3, 5, 10, 15, 20, 30, 40, 50]

for R in Rs:
    oracle = per_class_oracle_completeness(vals_sorted, R).mean().item()

    V_shared = global_vecs[:, :R]

    shared_diag = shared_diag_completeness_from_V(b, V_shared).mean().item()
    shared_full = shared_subspace_completeness_from_V(b, V_shared).mean().item()
    random = random_shared_diag_completeness(b, R, n_trials=20).mean().item()

    rows.append({"R": R, "completeness": oracle, "basis": "per-class eigenbasis"})
    rows.append({"R": R, "completeness": shared_diag, "basis": "shared diagonal basis"})
    rows.append({"R": R, "completeness": shared_full, "basis": "shared full subspace"})
    rows.append({"R": R, "completeness": random, "basis": "random diagonal basis"})

df = pd.DataFrame(rows)

px.line(
    df,
    x="R",
    y="completeness",
    color="basis",
    markers=True,
    template="plotly_white",
    title="Shared diagonal basis vs shared subspace decomposition",
)

#### next steps.

The failure of the shared diagonal basis suggests that the class matrices are not approximately jointly diagonalizable: there is no single set of independent visual eigenfeatures that all classes merely reweight. However, the success of the shared full-subspace reconstruction suggests that the class slices of B lie in a common low-dimensional visual subspace. A Tucker-style decomposition captures this by sharing the pixel-space factor V across classes while allowing each class to have its own small core matrix M
(c)
, which represents class-specific interactions among the shared visual features.

In [34]:
# b: [10, 784, 784]
b = 0.5 * (b + b.mT)
b = b.detach()

C, P, _ = b.shape

In [35]:
# Global interaction covariance
G = torch.zeros(P, P, device=b.device)

for c in range(C):
    G += b[c] @ b[c].T

G = 0.5 * (G + G.mT)

global_vals, global_vecs = torch.linalg.eigh(G)

# sort largest first
idx = torch.argsort(global_vals, descending=True)
global_vals = global_vals[idx]
global_vecs = global_vecs[:, idx]

# sanity check
print(
    "orthonormal error:",
    ((global_vecs.T @ global_vecs - torch.eye(P, device=b.device)).abs().max()).item()
)

orthonormal error: 2.0265579223632812e-06


In [36]:
def frob_sq(x):
    return x.pow(2).sum(dim=(-2, -1))


def tucker_shared_subspace(b, V):
    """
    b: [C, P, P]
    V: [P, R]

    returns:
        M:     [C, R, R]
        b_hat: [C, P, P]
        completeness: [C]
    """

    # M_c = V^T B_c V
    M = torch.einsum("ir,cij,js->crs", V, b, V)

    # B_c_hat = V M_c V^T
    b_hat = torch.einsum("ir,crs,js->cij", V, M, V)

    completeness = frob_sq(b_hat) / (frob_sq(b) + 1e-8)

    return M, b_hat, completeness

In [37]:
R = 20
V = global_vecs[:, :R]

M, b_hat, comp = tucker_shared_subspace(b, V)

print("per-class completeness:", comp.detach().cpu())
print("mean completeness:", comp.mean().item())

per-class completeness: tensor([0.9819, 0.9823, 0.9822, 0.9845, 0.9862, 0.9875, 0.9798, 0.9825, 0.9763,
        0.9800])
mean completeness: 0.9823273420333862


In [38]:
Rs = [1, 2, 3, 5, 10, 15, 20, 30, 40, 50]

rows = []

for R in Rs:
    V = global_vecs[:, :R]
    M, b_hat, comp = tucker_shared_subspace(b, V)

    rows.append({
        "R": R,
        "completeness": comp.mean().item(),
        "basis": "shared full subspace / Tucker"
    })

df_tucker = pd.DataFrame(rows)

px.line(
    df_tucker,
    x="R",
    y="completeness",
    markers=True,
    template="plotly_white",
    title="Tucker-style shared subspace reconstruction"
)

In [39]:
def shared_diagonal_completeness(b, V):
    """
    B_c ≈ sum_r a[c,r] v_r v_r^T
    """

    A = torch.einsum("ir,cij,jr->cr", V, b, V)

    b_hat = torch.einsum("cr,ir,jr->cij", A, V, V)

    completeness = frob_sq(b_hat) / (frob_sq(b) + 1e-8)

    return A, b_hat, completeness

In [40]:
rows = []

for R in Rs:
    V = global_vecs[:, :R]

    _, _, comp_diag = shared_diagonal_completeness(b, V)
    _, _, comp_full = tucker_shared_subspace(b, V)

    rows.append({
        "R": R,
        "completeness": comp_diag.mean().item(),
        "basis": "shared diagonal basis"
    })

    rows.append({
        "R": R,
        "completeness": comp_full.mean().item(),
        "basis": "shared full subspace / Tucker"
    })

df = pd.DataFrame(rows)

px.line(
    df,
    x="R",
    y="completeness",
    color="basis",
    markers=True,
    template="plotly_white",
    title="Shared diagonal basis vs Tucker-style shared subspace"
)

In [41]:
digit = 4
R = 20

V = global_vecs[:, :R]
M, b_hat, comp = tucker_shared_subspace(b, V)

px.imshow(
    M[digit].detach().cpu(),
    color_continuous_midpoint=0,
    color_continuous_scale="RdBu",
    title=f"Class-specific core M for digit {digit}"
)

In [42]:
import torch
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def tucker_components(V, M, offdiag_only=False):
    """
    V: [P, R] shared basis
    M: [C, R, R] class-specific Tucker cores

    Returns components of the form:
        activation = (v_r^T x)(v_s^T x)
        logits = class weights for this component
    """
    M = 0.5 * (M + M.mT)

    C, R, _ = M.shape
    comps = []

    for r in range(R):
        for s in range(r, R):
            if offdiag_only and r == s:
                continue

            logits = M[:, r, s].clone()

            # because M_rs and M_sr both contribute for r != s
            if r != s:
                logits = 2 * logits

            strength = logits.norm().item()

            comps.append({
                "r": r,
                "s": s,
                "left": V[:, r],
                "right": V[:, s],
                "plus": V[:, r] + V[:, s],
                "minus": V[:, r] - V[:, s],
                "logits": logits,
                "strength": strength,
            })

    comps = sorted(comps, key=lambda x: x["strength"], reverse=True)
    return comps

In [43]:
def summarize_components(comps, k=10):
    rows = []

    for idx, comp in enumerate(comps[:k]):
        logits = comp["logits"]

        top_pos = torch.argmax(logits).item()
        top_neg = torch.argmin(logits).item()

        rows.append({
            "rank": idx,
            "pair": f"({comp['r']}, {comp['s']})",
            "strength": comp["strength"],
            "top positive class": top_pos,
            "top positive logit": logits[top_pos].item(),
            "top negative class": top_neg,
            "top negative logit": logits[top_neg].item(),
        })

    return pd.DataFrame(rows)

In [44]:
R = 20
V = global_vecs[:, :R]
M, b_hat, comp = tucker_shared_subspace(b, V)

comps = tucker_components(V, M, offdiag_only=True)

summarize_components(comps, k=12)

,rank,pair,strength,top positive class,top positive logit,top negative class,top negative logit
0,0,"(0, 1)",1.331173,9,0.775321,5,-0.647127
1,1,"(1, 3)",1.252356,3,0.683631,4,-0.714706
2,2,"(0, 4)",1.213427,2,0.844918,5,-0.620480
3,3,"(0, 2)",1.150527,5,0.465916,4,-0.621942
4,4,"(1, 2)",1.068348,5,0.518983,0,-0.594614
5,5,"(2, 3)",1.030288,3,0.308397,4,-0.775942
6,6,"(2, 5)",0.971323,7,0.667897,3,-0.431757
7,7,"(0, 6)",0.961839,4,0.492489,7,-0.703346
8,8,"(0, 7)",0.872027,4,0.443889,5,-0.512118
9,9,"(1, 9)",0.870018,6,0.367808,2,-0.357083


In [47]:
def normalize_img(v, side=28):
    img = v.reshape(side, side)
    return img / (img.abs().max() + 1e-8)


def plot_tucker_components(comps, k=8, side=28):
    comps = comps[:k]

    fig = make_subplots(
        rows=3,
        cols=k,
        row_titles=["v_r + v_s", "v_r - v_s", "class logits"],
        vertical_spacing=0.08,
    )

    for col, comp in enumerate(comps, start=1):
        plus_img = normalize_img(comp["plus"], side).detach().cpu()
        minus_img = normalize_img(comp["minus"], side).detach().cpu()
        logits = comp["logits"].detach().cpu()

        fig.add_trace(
            go.Heatmap(
                z=plus_img,
                colorscale="RdBu",
                zmid=0,
                showscale=False,
            ),
            row=1,
            col=col,
        )

        fig.add_trace(
            go.Heatmap(
                z=minus_img,
                colorscale="RdBu",
                zmid=0,
                showscale=False,
            ),
            row=2,
            col=col,
        )

        fig.add_trace(
            go.Bar(
                x=list(range(len(logits))),
                y=logits,
                showlegend=False,
            ),
            row=3,
            col=col,
        )

        fig.add_annotation(
            text=f"({comp['r']},{comp['s']})<br>{comp['strength']:.2f}",
            xref=f"x{col}" if col > 1 else "x",
            yref=f"y{col}" if col > 1 else "y",
            x=side / 2,
            y=-2,
            showarrow=False,
            row=1,
            col=col,
        )

    for row in [1, 2]:
        for col in range(1, k + 1):
            fig.update_xaxes(visible=False, row=row, col=col)
            fig.update_yaxes(visible=False, row=row, col=col)

    fig.update_layout(
        width=140 * k,
        height=430,
        template="plotly_white",
        margin=dict(l=40, r=10, t=30, b=10),
    )

    fig.show()

In [48]:
plot_tucker_components(comps, k=8)

The Tucker decomposition reveals that the class slices of B share a common visual subspace, but class-specific computation is expressed through interactions between basis directions. Each column visualizes one such interaction: the first two rows show the constructive and contrastive visual patterns associated with the pair, while the final row shows how that interaction contributes to different digit logits. The dominance of off-diagonal pairs supports the view that the model does not merely reweight a shared set of independent eigenfeatures; instead, it combines shared visual directions differently for each class.

The model has shared visual ingredients, but digits are classified by combinations of ingredients, not by the ingredients independently.

In [49]:
# V: [P, R]
# M: [C, R, R] from tucker_shared_subspace(b, V)

M = 0.5 * (M + M.mT)

diag_vals = torch.diagonal(M, dim1=1, dim2=2)   # [C, R]

diag_energy = diag_vals.pow(2).sum(dim=1)       # [C]
total_energy = M.pow(2).sum(dim=(1, 2))         # [C]
offdiag_energy = total_energy - diag_energy

diag_frac = diag_energy / (total_energy + 1e-8)
offdiag_frac = offdiag_energy / (total_energy + 1e-8)

rows = []

for c in range(C):
    rows.append({
        "class": c,
        "fraction": diag_frac[c].item(),
        "core part": "diagonal"
    })
    rows.append({
        "class": c,
        "fraction": offdiag_frac[c].item(),
        "core part": "off-diagonal"
    })

df_core = pd.DataFrame(rows)

px.bar(
    df_core,
    x="class",
    y="fraction",
    color="core part",
    barmode="group",
    template="plotly_white",
    title="Diagonal vs off-diagonal energy in Tucker core",
    labels={"fraction": "fraction of Tucker core energy"}
)

To understand why the shared diagonal basis underperforms while the Tucker-style shared subspace succeeds, we measure the fraction of each class-specific core M
(c)
 lying on the diagonal versus off-diagonal entries. Diagonal entries correspond to independent squared shared features (v
r
⊤
	​

x)
2
, while off-diagonal entries correspond to interactions between distinct shared directions (v
r
⊤
	​

x)(v
s
⊤
	​

x). Across all digit classes, the majority of core energy is off-diagonal, indicating that class-specific computation is primarily implemented through interactions between shared visual directions rather than by independently reweighting a common set of eigenfeatures.

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import plotly.express as px

class SparseTuckerFeatures(nn.Module):
    def __init__(self, n_classes, R, K):
        super().__init__()
        self.U = nn.Parameter(torch.randn(R, K) * 0.02)
        self.A = nn.Parameter(torch.randn(n_classes, K) * 0.02)

    def forward(self):
        # normalize feature directions to remove scale degeneracy
        U = self.U / (self.U.norm(dim=0, keepdim=True) + 1e-8)

        # M_hat[c] = sum_k A[c,k] u_k u_k^T
        M_hat = torch.einsum("rk,ck,sk->crs", U, self.A, U)

        return M_hat, U, self.A

In [52]:
C, R, _ = M.shape
K = 64

torch.set_grad_enabled(True)

sparse_tucker = SparseTuckerFeatures(C, R, K).to(M.device)
opt = torch.optim.Adam(sparse_tucker.parameters(), lr=1e-2)

M_target = 0.5 * (M.detach() + M.detach().mT)

losses = []

with torch.enable_grad():
    for step in range(5000):
        M_hat, U, A = sparse_tucker()

        recon_loss = ((M_hat - M_target) ** 2).mean()
        sparsity_loss = A.abs().mean()

        loss = recon_loss + 1e-3 * sparsity_loss

        opt.zero_grad()
        loss.backward()
        opt.step()

        losses.append(loss.item())

px.line(y=losses, template="plotly_white", title="Sparse Tucker feature training loss")

In [53]:
M_hat, U, A = sparse_tucker()
loss = ((M_hat - M_target) ** 2).mean()

print("grad enabled:", torch.is_grad_enabled())
print("M_hat requires grad:", M_hat.requires_grad)
print("A requires grad:", A.requires_grad)
print("loss requires grad:", loss.requires_grad)

grad enabled: True
M_hat requires grad: True
A requires grad: True
loss requires grad: True


In [54]:
def frob_sq(x):
    return x.pow(2).sum(dim=(-2, -1))

with torch.no_grad():
    M_hat, U, A = sparse_tucker()

    # lift sparse core back to pixel space
    b_hat_sparse = torch.einsum("ir,crs,js->cij", V, M_hat, V)

    completeness = 1 - frob_sq(b - b_hat_sparse) / (frob_sq(b) + 1e-8)

print("per-class sparse completeness:", completeness.detach().cpu())
print("mean sparse completeness:", completeness.mean().item())

per-class sparse completeness: tensor([0.9798, 0.9795, 0.9806, 0.9823, 0.9850, 0.9864, 0.9778, 0.9807, 0.9734,
        0.9773])
mean sparse completeness: 0.9802879095077515


In [55]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

def plot_sparse_tucker_atoms(V, U, A, k=8, side=28):
    # visual atoms in pixel space
    atoms = V @ U  # [P, K]

    strengths = A.norm(dim=0)
    order = torch.argsort(strengths, descending=True)[:k]

    fig = make_subplots(
        rows=2,
        cols=k,
        row_titles=["visual atom", "class loading"],
        vertical_spacing=0.12,
    )

    for col, idx in enumerate(order, start=1):
        atom = atoms[:, idx]
        atom = atom / (atom.abs().max() + 1e-8)
        img = atom.reshape(side, side).detach().cpu()

        logits = A[:, idx].detach().cpu()

        fig.add_trace(
            go.Heatmap(
                z=img,
                colorscale="RdBu",
                zmid=0,
                showscale=False,
            ),
            row=1,
            col=col,
        )

        fig.add_trace(
            go.Bar(
                x=list(range(A.shape[0])),
                y=logits,
                showlegend=False,
            ),
            row=2,
            col=col,
        )

        fig.add_annotation(
            text=f"atom {idx.item()}<br>{strengths[idx].item():.2f}",
            xref=f"x{col}" if col > 1 else "x",
            yref=f"y{col}" if col > 1 else "y",
            x=side / 2,
            y=-2,
            showarrow=False,
            row=1,
            col=col,
        )

    for col in range(1, k + 1):
        fig.update_xaxes(visible=False, row=1, col=col)
        fig.update_yaxes(visible=False, row=1, col=col)

    fig.update_layout(
        width=140 * k,
        height=360,
        template="plotly_white",
        margin=dict(l=40, r=10, t=40, b=10),
    )

    fig.show()

In [56]:
with torch.no_grad():
    M_hat, U, A = sparse_tucker()

plot_sparse_tucker_atoms(V, U, A, k=8)

In [57]:
import torch
import pandas as pd
import plotly.express as px

def top_energy_fraction(v, frac=0.10, eps=1e-8):
    """
    Fraction of squared energy contained in the top frac pixels by magnitude.
    Higher = more localized.
    """
    flat = v.flatten()
    energy = flat.pow(2)
    k = max(1, int(frac * energy.numel()))
    top = torch.topk(energy, k=k).values.sum()
    total = energy.sum() + eps
    return top / total


def class_top_fraction(logits, eps=1e-8):
    """
    Fraction of class-loading energy in the strongest class.
    Higher = more class-selective.
    """
    energy = logits.pow(2)
    return energy.max() / (energy.sum() + eps)


def effective_support_fraction(v, eps=1e-8):
    """
    Effective number of active dimensions, normalized by dimensionality.
    Lower = sparser.
    """
    abs_v = v.abs()
    eff = abs_v.sum().pow(2) / (v.pow(2).sum() + eps)
    return eff / v.numel()


def effective_class_fraction(logits, eps=1e-8):
    """
    Effective number of active classes, normalized by number of classes.
    Lower = more class-sparse.
    """
    abs_l = logits.abs()
    eff = abs_l.sum().pow(2) / (logits.pow(2).sum() + eps)
    return eff / logits.numel()

In [58]:
with torch.no_grad():
    M_hat, U, A = sparse_tucker()

    atoms = V @ U   # [P, K]

rows = []

K = atoms.shape[1]

for k in range(K):
    atom = atoms[:, k]
    logits = A[:, k]

    rows.append({
        "component": k,
        "type": "sparse atom",
        "visual_top10_energy": top_energy_fraction(atom, frac=0.10).item(),
        "visual_effective_support": effective_support_fraction(atom).item(),
        "class_top_energy": class_top_fraction(logits).item(),
        "class_effective_support": effective_class_fraction(logits).item(),
        "strength": logits.norm().item(),
    })

df_sparse_metrics = pd.DataFrame(rows)

In [59]:
# assuming comps = tucker_components(V, M, offdiag_only=True)
# from earlier

rows = []

for idx, comp in enumerate(comps):
    # compare both plus and minus patterns by averaging their metrics
    plus = comp["plus"]
    minus = comp["minus"]
    logits = comp["logits"]

    visual_top10 = 0.5 * (
        top_energy_fraction(plus, frac=0.10)
        + top_energy_fraction(minus, frac=0.10)
    )

    visual_eff = 0.5 * (
        effective_support_fraction(plus)
        + effective_support_fraction(minus)
    )

    rows.append({
        "component": idx,
        "type": "Tucker pair",
        "visual_top10_energy": visual_top10.item(),
        "visual_effective_support": visual_eff.item(),
        "class_top_energy": class_top_fraction(logits).item(),
        "class_effective_support": effective_class_fraction(logits).item(),
        "strength": comp["strength"],
    })

df_tucker_metrics = pd.DataFrame(rows)

In [60]:
top_n = 64

df_compare = pd.concat([
    df_sparse_metrics.sort_values("strength", ascending=False).head(top_n),
    df_tucker_metrics.sort_values("strength", ascending=False).head(top_n),
])

px.box(
    df_compare,
    x="type",
    y="visual_top10_energy",
    points="all",
    template="plotly_white",
    title="Visual localization: fraction of energy in top 10% pixels",
)

In [61]:
px.box(
    df_compare,
    x="type",
    y="class_top_energy",
    points="all",
    template="plotly_white",
    title="Class selectivity: fraction of loading energy in strongest class",
)

In [62]:
px.box(
    df_compare,
    x="type",
    y="visual_effective_support",
    points="all",
    template="plotly_white",
    title="Visual effective support; lower means more localized",
)

In [63]:
px.box(
    df_compare,
    x="type",
    y="class_effective_support",
    points="all",
    template="plotly_white",
    title="Class effective support; lower means more class-selective",
)

We compare sparse Tucker atoms against raw Tucker pair components using localization and class-selectivity metrics. Sparse atoms concentrate more visual energy in a small number of pixels and place more class-loading energy on fewer classes. This suggests that sparse featurization is not necessary for faithful reconstruction, since the Tucker subspace already reconstructs B, but it is useful for interpretability: it converts distributed pairwise interactions into more localized, class-selective visual atoms.

In [ ]:
### story.
# 1. Per-class eigenbasis: faithful but class-specific.
# 2. Cross-class completeness: classes share a broader visual subspace.
# 3. Shared diagonal basis: fails, so shared independent eigenfeatures are insufficient.
# 4. Tucker/shared full subspace: succeeds, so classes share a low-dimensional visual space.
# 5. Off-diagonal core dominates: class-specific behavior comes from interactions between shared directions.
# 6. Sparse featurization: not needed for faithfulness, but improves interpretability by producing localized, class-selective atoms.